In [1]:
import pandas as pd
import math
import numpy as np
import btrack
from skimage.io import imread
from skimage.util import montage
from skimage.transform import resize
from IPython.display import HTML
from tifffile import imread
from tifffile import imwrite
import time
import json

In [5]:
segments = '/scratch/indikar_root/indikar1/shared_data/hybrid_imaging/segments'
props = '/scratch/indikar_root/indikar1/shared_data/hybrid_imaging/regionprops'

CONFIG_FILE = "cell_config.json"

In [3]:
%%time

control_name1 = '01_control_C2'
control_name2 = '02_control_C2'

control_segments1 = f'{segments}/{control_name1}.npy'
control_segments2 = f'{segments}/{control_name2}.npy'

control_segments_data1 = np.load(control_segments1)
control_segments_data2 = np.load(control_segments2)

print(f'{control_name1} segments shape = {control_segments_data1.shape}')
print(f'{control_name2} segments shape = {control_segments_data2.shape}')

01_control_C2 segments shape = (67, 2621, 3026)
02_control_C2 segments shape = (137, 2619, 3027)
CPU times: user 37.1 ms, sys: 3.18 s, total: 3.22 s
Wall time: 1min 49s


In [4]:
FEATURES = [
    "area", 
    "axis_major_length",
    "axis_minor_length", 
    "orientation", 
    "solidity",
    "eccentricity"
]

control_objects1 = btrack.utils.segmentation_to_objects(
    control_segments_data1, 
    properties=tuple(FEATURES), 
    num_workers=8,  # parallelise this
)

control_objects2 = btrack.utils.segmentation_to_objects(
    control_segments_data2, 
    properties=tuple(FEATURES), 
    num_workers=8,  # parallelise this
)

print('Finishing building track objects.')

[INFO][2026/07/27 05:06:22 PM] Localizing objects from segmentation...
[INFO][2026/07/27 05:06:22 PM] Processing using 8 workers.
100%|██████████| 67/67 [00:07<00:00,  8.79it/s]
[INFO][2026/07/27 05:06:30 PM] Objects are of type: <class 'dict'>
[INFO][2026/07/27 05:06:30 PM] ...Found 39967 objects in 67 frames.
[INFO][2026/07/27 05:06:30 PM] Localizing objects from segmentation...
[INFO][2026/07/27 05:06:30 PM] Processing using 8 workers.
100%|██████████| 137/137 [00:14<00:00,  9.62it/s]
[INFO][2026/07/27 05:06:45 PM] Objects are of type: <class 'dict'>
[INFO][2026/07/27 05:06:46 PM] ...Found 94187 objects in 137 frames.


Finishing building track objects.


In [6]:
n_frames_total, height, width = control_segments_data1.shape

with btrack.BayesianTracker() as tracker:
    tracker.configure(CONFIG_FILE)
    tracker.max_search_radius = 50
    tracker.tracking_updates = ["MOTION", "VISUAL"]
    tracker.features = FEATURES
    tracker.append(control_objects1)
    tracker.volume = ((0, width), (0, height))
    tracker.track(step_size=100)
    tracker.optimize()
    tracks = tracker.tracks

[INFO][2026/07/27 05:12:08 PM] Loaded btrack: /nfs/turbo/umms-indikar/Jillian/conda-envs/btrack-env/lib/python3.11/site-packages/btrack/libs/libtracker.so
[INFO][2026/07/27 05:12:08 PM] Starting BayesianTracker session
[INFO][2026/07/27 05:12:08 PM] Loading configuration file: cell_config.json
[INFO][2026/07/27 05:12:08 PM] Objects are of type: <class 'list'>
[INFO][2026/07/27 05:12:09 PM] Starting tracking... 
[INFO][2026/07/27 05:12:09 PM] Update using: ['MOTION', 'VISUAL']
[INFO][2026/07/27 05:12:09 PM] Tracking objects in frames 0 to 67 (of 67)...
[INFO][2026/07/27 05:12:33 PM]  - Timing (Bayesian updates: 148.08ms, Linking: 3.89ms)
[INFO][2026/07/27 05:12:33 PM]  - Probabilities (Link: 1.00000, Lost: 1.00000)
[INFO][2026/07/27 05:12:33 PM] SUCCESS.
[INFO][2026/07/27 05:12:33 PM]  - Found 1267 tracks in 67 frames (in 0.0s)
[INFO][2026/07/27 05:12:33 PM]  - Inserted 553 dummy objects to fill tracking gaps
[INFO][2026/07/27 05:12:33 PM] Loading hypothesis model: cell_hypothesis
[INFO

GLPK Integer Optimizer, v4.65
5068 rows, 3908 columns, 5289 non-zeros
3908 integer variables, all of which are binary
Preprocessing...
2534 rows, 3908 columns, 5289 non-zeros
3908 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 2534
Solving LP relaxation...
GLPK Simplex Optimizer, v4.65
2534 rows, 3908 columns, 5289 non-zeros
*     0: obj =   7.553984313e+03 inf =   0.000e+00 (570)
*   571: obj =   3.088938285e+03 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Integer optimization begins...
Long-step dual simplex will be used
+   571: mip =     not found yet >=              -inf        (1; 0)
+   571: >>>>>   3.088938285e+03 >=   3.088938285e+03   0.0% (1; 0)
+   571: mip =   3.088938285e+03 >=     tree is empty   0.0% (0; 1)
INTEGER OPTIMAL SOLUTION FOUND


In [12]:
lengths = np.array([len(t) for t in tracks])

print(f"n_tracks     = {len(lengths)}")
print(f"mean_len     = {lengths.mean():.2f}")
print(f"median_len   = {np.median(lengths):.1f}")
print(f"frac_len1    = {(lengths == 1).mean():.2%}")
print(f"frac_len_lt5 = {(lengths < 5).mean():.2%}")
print(f"frac_len_ge20= {(lengths >= 20).mean():.2%}")
print(f"max_len      = {lengths.max()}  (out of {n_frames_total} frames)")

n_tracks     = 1203
mean_len     = 33.56
median_len   = 24.0
frac_len1    = 30.34%
frac_len_lt5 = 37.82%
frac_len_ge20= 51.54%
max_len      = 67  (out of 67 frames)


In [13]:
n_frames_total, height, width = control_segments_data2.shape

with btrack.BayesianTracker() as tracker:
    tracker.configure(CONFIG_FILE)
    tracker.max_search_radius = 50
    tracker.tracking_updates = ["MOTION", "VISUAL"]
    tracker.features = FEATURES
    tracker.append(control_objects2)
    tracker.volume = ((0, width), (0, height))
    tracker.track(step_size=100)
    tracker.optimize()
    tracks2 = tracker.tracks

[INFO][2026/07/27 05:16:57 PM] Loaded btrack: /nfs/turbo/umms-indikar/Jillian/conda-envs/btrack-env/lib/python3.11/site-packages/btrack/libs/libtracker.so
[INFO][2026/07/27 05:16:57 PM] Starting BayesianTracker session
[INFO][2026/07/27 05:16:57 PM] Loading configuration file: cell_config.json
[INFO][2026/07/27 05:16:57 PM] Objects are of type: <class 'list'>
[INFO][2026/07/27 05:16:58 PM] Starting tracking... 
[INFO][2026/07/27 05:16:58 PM] Update using: ['MOTION', 'VISUAL']
[INFO][2026/07/27 05:16:59 PM] Tracking objects in frames 0 to 99 (of 137)...
[INFO][2026/07/27 05:17:47 PM]  - Timing (Bayesian updates: 287.53ms, Linking: 5.43ms)
[INFO][2026/07/27 05:17:47 PM]  - Probabilities (Link: 1.00000, Lost: 0.97890)
[INFO][2026/07/27 05:17:47 PM]  - Stats (Active: 868, Lost: 1975, Conflicts resolved: 7204)
[INFO][2026/07/27 05:17:47 PM] Tracking objects in frames 100 to 137 (of 137)...
[INFO][2026/07/27 05:18:19 PM]  - Timing (Bayesian updates: 416.90ms, Linking: 12.13ms)
[INFO][2026/07

GLPK Integer Optimizer, v4.65
13668 rows, 11452 columns, 16232 non-zeros
11452 integer variables, all of which are binary
Preprocessing...
6834 rows, 11452 columns, 16232 non-zeros
11452 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 6834
Solving LP relaxation...
GLPK Simplex Optimizer, v4.65
6834 rows, 11452 columns, 16232 non-zeros
*     0: obj =   3.172935564e+04 inf =   0.000e+00 (2744)
Perturbing LP to avoid stalling [1658]...
Removing LP perturbation [2670]...
*  2670: obj =   1.234438549e+04 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Integer optimization begins...
Long-step dual simplex will be used
+  2670: mip =     not found yet >=              -inf        (1; 0)
+  2678: >>>>>   1.235486902e+04 >=   1.235265115e+04 < 0.1% (9; 0)
+  2685: mip =   1.235486902e+04 >=     tree is empty   0.0% (0; 17)
INTEGER

In [14]:
lengths = np.array([len(t) for t in tracks2])

print(f"n_tracks     = {len(lengths)}")
print(f"mean_len     = {lengths.mean():.2f}")
print(f"median_len   = {np.median(lengths):.1f}")
print(f"frac_len1    = {(lengths == 1).mean():.2%}")
print(f"frac_len_lt5 = {(lengths < 5).mean():.2%}")
print(f"frac_len_ge20= {(lengths >= 20).mean():.2%}")
print(f"max_len      = {lengths.max()}  (out of {n_frames_total} frames)")

n_tracks     = 2899
mean_len     = 32.93
median_len   = 5.0
frac_len1    = 37.84%
frac_len_lt5 = 49.26%
frac_len_ge20= 35.05%
max_len      = 137  (out of 137 frames)
